# Iceberg Write + PG Read/Search

**Routing pattern:**
- WRITE: `[iceberg, postgresql]` — Iceberg is primary (ACID + time-travel); PG secondary for operational queries
- READ: `[postgresql]` — fast operational reads
- SEARCH: `[postgresql]` — SQL-powered attribute + spatial queries
- METADATA: `[postgresql]` — PG is always the metadata authority

Hybrid pattern: Iceberg for analytics/versioning, PG for operational API serving. Best when you need both time-travel history and sub-millisecond single-item lookups.

In [ ]:
import asyncio
import httpx

import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True), override=False)
BASE = os.environ.get("DYNASTORE_BASE_URL") or "http://localhost:8080"

# Auto-provision DYNASTORE_TOKEN via IDP client_credentials if not already set
if not os.environ.get("DYNASTORE_TOKEN"):
    _idp = (os.environ.get("IDP_PUBLIC_URL") or os.environ.get("IDP_ISSUER_URL", "")).rstrip("/")
    _cid = os.environ.get("IDP_CLIENT_ID", "")
    _sec = os.environ.get("IDP_CLIENT_SECRET", "")
    if _idp and _cid and _sec:
        try:
            _r = httpx.post(
                f"{_idp}/protocol/openid-connect/token",
                data={"grant_type": "client_credentials", "client_id": _cid, "client_secret": _sec},
                timeout=10,
            )
            if _r.status_code == 200:
                os.environ["DYNASTORE_TOKEN"] = _r.json().get("access_token", "")
        except Exception:
            pass
CATALOG_ID = "demo-hybrid"
COLLECTION_ID = "sentinel-scenes"

client = httpx.AsyncClient(base_url=BASE, timeout=30)

def _check(r, label=""):
    status = r.status_code
    ok = status < 400
    suffix = "" if ok else f" — {r.text[:300]}"
    print(f"{label + ': ' if label else ''}{status}{suffix}")
    return r

print("Client ready")

In [ ]:
r = await client.post("/stac/catalogs", json={"id": CATALOG_ID, "title": "Hybrid Demo", "description": "Hybrid Demo catalog."})
_check(r)
# Wait for provisioning to complete before creating collections
for _ in range(30):
    r = await client.get(f"/stac/catalogs/{CATALOG_ID}")
    if r.status_code == 200 and r.json().get("provisioning_status") == "ready":
        print("Catalog ready")
        break
    await asyncio.sleep(1)
else:
    print("Warning: catalog still provisioning after 30s")

In [ ]:
r = await client.post(f"/stac/catalogs/{CATALOG_ID}/collections", json={
    "id": COLLECTION_ID,
    "title": "Sentinel-2 Scenes",
    "description": "EO scenes: Iceberg for versioned archive, PG for fast API",
    "license": "proprietary",
    "extent": {"spatial": {"bbox": [[-180, -90, 180, 90]]}, "temporal": {"interval": [["2020-01-01T00:00:00Z", None]]}},
})
_check(r)

In [ ]:
routing = {
    "operations": {
        "WRITE": [
            {"driver_ref": "items_iceberg_driver", "on_failure": "fatal", "write_mode": "sync"},
            {"driver_ref": "items_postgresql_driver", "on_failure": "warn", "write_mode": "sync"}
        ],
        "READ": [{"driver_ref": "items_postgresql_driver"}],
        "SEARCH": [{"driver_ref": "items_postgresql_driver"}],
        "METADATA": [{"driver_ref": "items_postgresql_driver"}]
    }
}
r = await client.put(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/plugins/collection_routing_config",
    json=routing,
)
_check(r)

In [ ]:
# UPDATE policy: overwrite in-place when re-ingesting same scene ID
policy = {
    "on_conflict": "update",
    "track_asset_id": True,
    "validity": {"column": "valid_from"},
    "external_id_field": "id"
}
r = await client.put(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/plugins/items_write_policy",
    json=policy,
)
_check(r)

In [ ]:
import random, datetime

features = [
    {
        "type": "Feature",
        "id": f"S2A_MSIL2A_2024{i:03d}",
        "geometry": {
            "type": "Polygon",
            "coordinates": [[[10+i, 40+i], [11+i, 40+i], [11+i, 41+i], [10+i, 41+i], [10+i, 40+i]]]
        },
        "bbox": [10+i, 40+i, 11+i, 41+i],
        "properties": {
            "platform": "sentinel-2a",
            "cloud_cover": round(random.uniform(0, 30), 1),
            "datetime": f"2024-01-{i+1:02d}T10:00:00Z",
            "processing_level": "L2A",
        }
    }
    for i in range(5)
]
r = await client.post(
    f"/stac/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items",
    json={"type": "FeatureCollection", "features": features},
)
_check(r, "Scenes written to both Iceberg + PG")

In [ ]:
# READ from PG (operational, fast)
r = await client.get(f"/stac/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items?limit=10")
data = r.json()
print(f"PG READ: {len(data.get('features', []))} scenes")
for f in data.get("features", []):
    p = f["properties"]
    print(f"  {f['id']} cloud={p.get('cloud_cover')}%")

In [ ]:
# SEARCH from PG — filter by cloud cover < 10%
r = await client.post(
    f"/search/catalogs/{CATALOG_ID}",
    json={
        "collections": [COLLECTION_ID],
        "filter": {"op": "<", "args": [{"property": "cloud_cover"}, 10]},
        "limit": 10,
    },
)
data = r.json()
print(f"Clear scenes (cloud < 10%): {len(data.get('features', []))}")

In [ ]:
# Verify Iceberg received the same data (via admin endpoint or direct PyIceberg)
# In production, use time-travel to query a previous snapshot
print("Iceberg table now holds a snapshot with all 5 scenes.")
print("Time-travel example (requires PyIceberg direct access):")
print("""
  from pyiceberg.catalog import load_catalog
  cat = load_catalog('pg', uri='postgresql+psycopg2://...')
  tbl = cat.load_table(('demo_hybrid', 'sentinel_scenes'))
  # List snapshots
  for snap in tbl.metadata.snapshots:
      print(snap.snapshot_id, snap.timestamp_ms)
  # Time-travel read
  old_data = tbl.scan(snapshot_id=tbl.metadata.snapshots[0].snapshot_id).to_arrow()
""")

In [ ]:
r = await client.delete(f"/stac/catalogs/{CATALOG_ID}?force=true")
_check(r, "Cleanup")
await client.aclose()